# 02 · Construir el pipeline

Este cuaderno es un **taller**. En cada paso escribís vos una pieza, un
*checkpoint* te dice si va bien, y si te trabás hay una solución plegada.

Trabajamos sobre el caso de ejemplo que viene en el repo. En `03_tu_caso`
cambiamos el caso por el tuyo.

### Cómo usar cada paso

1. **El problema** — qué hay que resolver.
2. **Tu turno** — completás una celda con `# TODO`.
3. **Checkpoint** — corré la celda siguiente: te dice qué falla y qué no.
4. **🔑 Solución** — plegada, ábrela solo si te trabaste.
5. **🤖 Pedíselo a Claude** — el prompt que conviene usar. Esta caja es la mitad
   del curso: el objetivo no es que memorices regex, es que sepas **qué pedir**.

> ⚠️ Los pasos 1 a 11 corren en Colab. **El paso 12 (la carga) es local**, porque
> vas a querer ver el navegador trabajando en tiempo real.

## Preparar el entorno

In [ ]:
# Funciona en Colab y en local. Si ya estás dentro del repo, no hace nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/GEJ1/data_entry_automatizado.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if not Path("pipeline/contratos.py").exists():
    if Path("data_entry_automatizado/pipeline/contratos.py").exists():
        os.chdir("data_entry_automatizado")
    else:
        print("Clonando el repo...")
        subprocess.run(["git", "clone", "-q", REPO], check=True)
        os.chdir("data_entry_automatizado")

if en_colab():
    print("Instalando dependencias (un par de minutos la primera vez)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "requirements.txt"], check=True)

PY = sys.executable

def correr(comando, mostrar=True):
    """Corre un comando del proyecto y muestra su salida."""
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    salida = (r.stdout + r.stderr).rstrip()
    if mostrar:
        print(salida)
    return salida

def cli(args, mostrar=True):
    return correr(f"{PY} -m pipeline.cli {args}", mostrar)

print("\nlisto ·", Path.cwd(), "· python", sys.version.split()[0])

In [ ]:
def checkpoint(titulo, casos, fn):
    """Corre tu función contra casos conocidos y te dice cómo vas."""
    print(titulo)
    print("-" * len(titulo))
    ok = 0
    for entrada, esperado in casos:
        args = entrada if isinstance(entrada, tuple) else (entrada,)
        try:
            got = fn(*args)
        except Exception as e:
            got = f"ERROR: {type(e).__name__}: {e}"
        bien = got == esperado
        ok += bien
        marca = "OK " if bien else "MAL"
        extra = "" if bien else f"   ← esperaba {esperado!r}"
        print(f"  [{marca}] {entrada!r} → {got!r}{extra}")
    total = len(casos)
    print(f"\n  {ok}/{total} " + ("✅ listo, seguí" if ok == total else "⏳ todavía no"))
    return ok == total

print("checkpoint() cargado")

---
# Paso 1 · El contrato

Antes de escribir una línea de parser hay que decidir **dónde están las
junturas**, o vas a terminar con la lógica de negocio metida adentro del código
que abre PDFs. El día que llegue un DOCX, tirás todo.

La regla que ordena todo el proyecto:

> **El extractor no sabe de negocio.** Devuelve texto crudo en cabecera y tablas.
> Toda interpretación vive en el dominio.

## Tu turno

Para cada responsabilidad, decidí de quién es: `"extractor"` o `"dominio"`.

In [ ]:
mis_respuestas = {
    "abrir el archivo y encontrar las tablas":      "?",
    "convertir '23,000' en el número 23000":        "?",
    "saber que una ficha sigue en la página que viene": "?",
    "decidir que un CUIT con verificador malo se carga igual": "?",
    "colapsar los saltos de línea de una celda angosta": "?",
    "resolver que '2 meses' es una fecha":           "?",
    "descartar las filas marcadas como inactivas":   "?",
}

In [ ]:
import base64, json

# Las respuestas van codificadas para que no te las lleves puestas al leer
# la celda. Corré nomás: el checkpoint te dice cuáles acertaste.
_CLAVE = "eyJhYnJpciBlbCBhcmNoaXZvIHkgZW5jb250cmFyIGxhcyB0YWJsYXMiOiAiZXh0cmFjdG9yIiwgImNvbnZlcnRpciAnMjMsMDAwJyBlbiBlbCBuw7ptZXJvIDIzMDAwIjogImRvbWluaW8iLCAic2FiZXIgcXVlIHVuYSBmaWNoYSBzaWd1ZSBlbiBsYSBww6FnaW5hIHF1ZSB2aWVuZSI6ICJleHRyYWN0b3IiLCAiZGVjaWRpciBxdWUgdW4gQ1VJVCBjb24gdmVyaWZpY2Fkb3IgbWFsbyBzZSBjYXJnYSBpZ3VhbCI6ICJkb21pbmlvIiwgImNvbGFwc2FyIGxvcyBzYWx0b3MgZGUgbMOtbmVhIGRlIHVuYSBjZWxkYSBhbmdvc3RhIjogImV4dHJhY3RvciIsICJyZXNvbHZlciBxdWUgJzIgbWVzZXMnIGVzIHVuYSBmZWNoYSI6ICJkb21pbmlvIiwgImRlc2NhcnRhciBsYXMgZmlsYXMgbWFyY2FkYXMgY29tbyBpbmFjdGl2YXMiOiAiZG9taW5pbyJ9"
ESPERADO = json.loads(base64.b64decode(_CLAVE).decode())

checkpoint("Paso 1 · de quién es cada responsabilidad",
           [(k, v) for k, v in ESPERADO.items()],
           lambda k: mis_respuestas.get(k, "?"))

La regla práctica: si para decidirlo necesitás saber **de qué se trata el
negocio**, es del dominio. Si te alcanza con saber cómo está armado el archivo,
es del extractor.

Los dos "extractor" que suelen sorprender —el derrame de página y el colapso de
saltos de línea— son mañas del *render*, no del dato. Un DOCX tiene otras
distintas, y por eso no pueden vivir en el dominio.

In [ ]:
from pipeline.contratos import FichaCruda
import inspect
print(inspect.getsource(FichaCruda)[:900])

> ### 🤖 Pedíselo a Claude
>
> ```
> Antes de escribir el parser, definí el contrato entre la extracción y
> el dominio para este proyecto: una estructura de datos que represente una unidad
> del documento SIN interpretar (solo texto), de forma que después pueda enchufar
> otro formato de entrada sin tocar el resto del pipeline.
> ```

Compará este prompt con *«hacé un parser de PDF»*. La diferencia entre los dos es todo el curso.

---
# Paso 2 · Datos falsos con trampas

No vas a desarrollar contra datos reales: son sensibles, no los podés commitear,
y encima no tenés control sobre qué casos raros aparecen.

La jugada es **generar datos falsos que incluyan a propósito cada caso difícil**.

In [ ]:
correr(f"{PY} demo/generar_pdf_fake.py")

Cada línea del manifiesto es una decisión de diseño esperando ser tomada. Mirá
cómo se declaran:

In [ ]:
import inspect
from demo import datos_fake  # el generador vive en demo/
fuente = inspect.getsource(datos_fake.clientes_trampa)
print(fuente[fuente.index("# 3)"):fuente.index("# 6)")])

> ### 🤖 Pedíselo a Claude
>
> ```
> Generá un script que produzca PDFs falsos con la misma estructura que
> mi documento real, y que siembre a propósito los casos difíciles: valores en
> formatos inconsistentes, datos basura, secciones vacías, fichas que se derraman a
> varias páginas y filas duplicadas. Que imprima un manifiesto diciendo qué cliente
> muestra qué caso.
> ```

---
# Paso 3 · El ground truth

**Este es el paso que casi nadie hace y el que más te va a servir.**

El generador sabe exactamente qué escribió en el archivo. Entonces puede dejar
al lado un JSON con lo que un extractor correcto *tendría* que devolver.

## Tu turno

Dado un cliente del generador, armá la ficha esperada: cabecera + tablas.

Ojo con dos detalles que son justamente donde se equivoca todo el mundo:

- la columna `Solicitud` es **sintética**: en el documento es un título que
  abarca varias filas, y hay que bajarlo a cada fila;
- la emisión lleva la hora pegada, tal como aparece en el papel.

In [ ]:
from datetime import date

cliente = {
    "nombre": "ACME SA",
    "cuit": "30709204595",
    "emision": date(2026, 7, 3),
    "referencias": [
        {"titulo": "Solicitud hecha el 09/01/2025 por FREE",
         "filas": [{"Fecha": "12/5/2006", "Plazo": "30"},
                   {"Fecha": "01/02/2025", "Plazo": "3060"}]},
    ],
    "alertas": [{"Fecha": "10/01/2026", "Tipo": "Mora"}],
}

def mi_verdad(cli):
    """Devuelve {'cabecera': {...}, 'tablas': {'referencias': [...], 'alertas': [...]}}"""
    # TODO: armar la ficha esperada
    return {}

In [ ]:
ESPERADA = {
    "cabecera": {"Cliente": "ACME SA", "CUIT": "30709204595",
                 "Fecha de Emisión": "03/07/2026 19:00"},
    "tablas": {
        "referencias": [
            {"Solicitud": "Solicitud hecha el 09/01/2025 por FREE",
             "Fecha": "12/5/2006", "Plazo": "30"},
            {"Solicitud": "Solicitud hecha el 09/01/2025 por FREE",
             "Fecha": "01/02/2025", "Plazo": "3060"},
        ],
        "alertas": [{"Fecha": "10/01/2026", "Tipo": "Mora"}],
    },
}
checkpoint("Paso 3 · el ground truth", [(cliente, ESPERADA)], mi_verdad)

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_verdad(cli):
    referencias = []
    for sg in cli["referencias"]:
        for f in sg["filas"]:
            # La solicitud es un titulo que abarca varias filas: se baja a cada una
            referencias.append({"Solicitud": sg["titulo"], **f})
    return {
        "cabecera": {
            "Cliente": cli["nombre"],
            "CUIT": cli["cuit"],
            "Fecha de Emisión": cli["emision"].strftime("%d/%m/%Y") + " 19:00",
        },
        "tablas": {"referencias": referencias, "alertas": list(cli["alertas"])},
    }
```


En el repo: [`demo/datos_fake.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/demo/datos_fake.py)

</details>

Con eso, verificar el parser deja de ser mirar columnas a ojo:

In [ ]:
correr(f"{PY} -m tests.verificar_extractor data/entrada/lote17.pdf | tail -3")

### Qué se ve cuando algo se rompe

Le metemos dos errores a una ficha —un formato distinto y una fila que
desaparece— y miramos qué reporta:

In [ ]:
import copy, json
from pathlib import Path
from tests.verificar_extractor import _diferencias

verdad = json.loads(Path("data/entrada/lote17.verdad.json").read_text(encoding="utf-8"))
buena = verdad[0]
rota = copy.deepcopy(buena)
rota["tablas"]["referencias"][0]["Antigüedad"] = "12/5/06"   # otro formato
del rota["tablas"]["referencias"][1]                          # se come una fila

for d in _diferencias(buena, rota):
    print(" ·", d)

> ### 🤖 Pedíselo a Claude
>
> ```
> Hacé que el generador de datos falsos escriba también un archivo con
> el "ground truth": exactamente lo que un extractor correcto debería devolver para
> ese documento. Y un script que compare la salida del extractor contra ese archivo
> y muestre las diferencias campo por campo.
> ```

---
# Paso 4 · La columna sucia

Acá empieza el trabajo real, que no es técnico: es **decidir**.

Mirá la columna *Antigüedad* del cliente 2. Diez formas de escribir lo mismo, en
la misma columna, cargadas por personas distintas a lo largo de años:

`12/5/2006` · `05/2019` · `05/18` · `2011` · `2 meses` · `1 año` · `3` ·
`reciente` · `-` · `n/c`

Las reglas que vamos a aplicar, en orden de prioridad (gana el primer patrón que
matchea):

| Patrón | Interpretación |
|---|---|
| `d/m/aaaa` | fecha completa |
| `m/aaaa` | día = 01 |
| `m/aa` | día = 01, siglo con pivote en 26 (00–26 ⇒ 20xx, 27–99 ⇒ 19xx) |
| `aaaa` (4 dígitos) | año solo ⇒ 01/07/aaaa |
| `N meses` / `N años` | se resta **a la fecha de emisión** |
| 1–3 dígitos | cantidad de años ⇒ emisión − N |
| palabra sola, `-`, `n/c` | sin fecha ⇒ `None` |

**La decisión más importante está en la fila de los relativos.** `2 meses` ¿desde
cuándo? Si lo anclás a `date.today()`, el mismo documento da resultados distintos
según el día que lo corras. Se ancla a la **fecha de emisión**: el documento pasa
a ser una función pura.

## Tu turno

In [ ]:
import re
from datetime import date
from dateutil.relativedelta import relativedelta

VACIOS = {"", "-", "n/c", "nc", "reciente"}

def mi_parse_antiguedad(raw, emision):
    """Devuelve un date, o None si no hay fecha interpretable."""
    if raw is None:
        return None
    s = raw.strip().lower()
    # TODO: implementar el matcher de prioridad
    return None

In [ ]:
EMISION = date(2026, 7, 3)
casos = [
    (("12/5/2006", EMISION), date(2006, 5, 12)),
    (("05/2019", EMISION),   date(2019, 5, 1)),
    (("05/18", EMISION),     date(2018, 5, 1)),
    (("05/98", EMISION),     date(1998, 5, 1)),
    (("2011", EMISION),      date(2011, 7, 1)),
    (("2 meses", EMISION),   date(2026, 5, 3)),
    (("1 año", EMISION),     date(2025, 7, 3)),
    (("3", EMISION),         date(2023, 7, 3)),
    (("18", EMISION),        date(2008, 7, 3)),
    (("reciente", EMISION),  None),
    (("-", EMISION),         None),
    (("n/c", EMISION),       None),
]
checkpoint("Paso 4 · antigüedad", casos, mi_parse_antiguedad)

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_parse_antiguedad(raw, emision):
    if raw is None:
        return None
    s = raw.strip().lower()
    if s in VACIOS:
        return None

    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)      # d/m/aaaa
    if m:
        d, mo, y = map(int, m.groups())
        return date(y, mo, d)

    m = re.fullmatch(r"(\d{1,2})/(\d{4})", s)                 # m/aaaa
    if m:
        mo, y = map(int, m.groups())
        return date(y, mo, 1)

    m = re.fullmatch(r"(\d{1,2})/(\d{2})", s)                 # m/aa
    if m:
        mo, yy = map(int, m.groups())
        anio = 2000 + yy if yy <= 26 else 1900 + yy            # pivote
        return date(anio, mo, 1)

    m = re.fullmatch(r"(\d{4})", s)                            # aaaa
    if m:
        return date(int(m.group(1)), 7, 1)

    m = re.match(r"(\d+)\s*(mes|año|anio)", s)                # relativos
    if m:
        n = int(m.group(1))
        if s[m.start(2):].startswith("mes"):
            return emision - relativedelta(months=n)
        return emision - relativedelta(years=n)

    m = re.fullmatch(r"(\d{1,3})", s)                          # N años
    if m:
        return emision - relativedelta(years=int(m.group(1)))

    return None
```


En el repo: [`pipeline/dominio/normalizadores.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/pipeline/dominio/normalizadores.py)

</details>

> **El orden importa.** Si probaras `1-3 dígitos ⇒ años` antes que `aaaa`, el
> valor `2011` se leería como "2011 años atrás". Por eso es un matcher de
> prioridad y no un `if` suelto.

> ### 🤖 Pedíselo a Claude
>
> ```
> En esta columna los valores vienen en formatos inconsistentes:
> [pegá 10 ejemplos reales]. Escribí un normalizador con un matcher de prioridad,
> documentando qué patrón gana sobre cuál y por qué. Los valores relativos ("2
> meses") tienen que anclarse a la fecha de emisión del documento y NO a
> date.today(), para que el mismo documento dé siempre el mismo resultado.
> ```

---
# Paso 5 · Qué hacer con lo que no se puede interpretar

Dos casos, dos decisiones distintas. Y ninguna es "tirar una excepción".

**Plazo:** son cuotas, válido de 0 a 90. En los datos aparece `3060` (alguien
escribió dos plazos pegados) y `150`. ¿Qué hacés? → El campo queda en `None`,
se marca como problema, **pero la fila igual se carga**. El resto de sus datos
sirve.

**CUIT:** es la clave del cliente, con dígito verificador. Si el verificador
está mal, ¿bloqueás el cliente? → **No.** Se carga y se marca. Un cliente nunca
puede desaparecer en silencio.

## Tu turno

In [ ]:
def mi_parse_plazo(raw):
    """Devuelve (valor, es_problema). Válido: entero 0-90."""
    # TODO
    return (None, False)


def mi_cuit_valido(cuit):
    """11 dígitos + dígito verificador (mod 11)."""
    # TODO
    return False

In [ ]:
checkpoint("Paso 5a · plazo", [
    ("30", (30, False)), ("0", (0, False)), ("90", (90, False)),
    ("3060", (None, True)), ("150", (None, True)),
    ("", (None, False)), ("-", (None, False)), ("treinta", (None, True)),
], mi_parse_plazo)

print()
checkpoint("Paso 5b · CUIT", [
    ("30-70920459-5", True), ("30709204595", True),
    ("30-70920459-4", False), ("123", False),
], mi_cuit_valido)

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_parse_plazo(raw):
    if raw is None:
        return (None, False)
    s = raw.strip()
    if s in ("", "-"):
        return (None, False)      # vacio no es un problema, es un vacio
    if not s.isdigit():
        return (None, True)
    v = int(s)
    return (v, False) if 0 <= v <= 90 else (None, True)


def mi_cuit_valido(cuit):
    d = re.sub(r"\D", "", cuit or "")
    if len(d) != 11:
        return False
    pesos = [5, 4, 3, 2, 7, 6, 5, 4, 3, 2]
    resto = sum(int(x) * w for x, w in zip(d[:10], pesos)) % 11
    verif = 0 if resto == 0 else (9 if resto == 1 else 11 - resto)
    return verif == int(d[10])
```


En el repo: [`pipeline/dominio/normalizadores.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/pipeline/dominio/normalizadores.py)

</details>

Fijate la diferencia entre `""` y `"treinta"`: el primero es un **vacío legítimo**
(no hay dato) y el segundo es un **problema** (hay dato y no se entiende). Meterlos
en la misma bolsa te esconde errores reales entre cientos de celdas vacías.

---
# Paso 6 · El extractor

Ahora sí, abrir el PDF. Tres problemas, y ninguno es "leer una tabla".

### Problema 1 · Las fichas se derraman

Un cliente con muchas filas sigue en la página siguiente, y esa página **no
repite el encabezado del cliente**. Mirá:

In [ ]:
import pdfplumber
pdf = pdfplumber.open("data/entrada/lote17.pdf")

print("página 9 — arranca un cliente:")
for l in pdf.pages[8].extract_text_lines()[:2]:
    print("   ", l["text"][:65])

print("\npágina 10 — es la continuación, pero no lo dice:")
for l in pdf.pages[9].extract_text_lines()[:2]:
    print("   ", l["text"][:65])

Por eso **nunca se procesa página por página**: se streamea el documento entero
y se corta por el encabezado `Cliente: ... (CUIT: ...)`.

### Problema 2 · Qué tabla es cuál

En la página de continuación tampoco está el título de sección (*"1. Solicitudes
de Referencia"*). Lo único que se repite son los **encabezados de columna**. Así
que la sección se decide mirando la tabla misma.

### Problema 3 · Filas que no son filas

El título de subgrupo (*"Solicitud hecha el X por Y"*) es una celda combinada.
Hay que detectarla y bajarla a las filas que le siguen, porque **la solicitud es
parte de la clave** que identifica cada registro.

In [ ]:
fila = pdf.pages[0].find_tables()[0].extract()[1]
print("así llega una fila de subgrupo:")
print("  ", fila[:4], "...")

## Tu turno

In [ ]:
from pipeline.dominio.normalizadores import clave_columna

def mi_seccion_de(encabezados):
    """'referencias', 'alertas' o None, mirando los encabezados de la tabla."""
    # TODO: pista — usá clave_columna() para tolerar tildes y mayúsculas
    return None


def mi_es_subgrupo(celdas):
    """True si esta fila es un título de subgrupo y no datos.
    `celdas` ya viene con los None convertidos a ''. """
    # TODO
    return False

In [ ]:
checkpoint("Paso 6a · qué tabla es", [
    (["Fecha", "Informante", "¿Es cliente?", "CO ($)"], "referencias"),
    (["Fecha", "Alertante", "Tipo", "Estado"], "alertas"),
    (["Fecha", "Cliente", "Total"], None),
], mi_seccion_de)

print()
checkpoint("Paso 6b · fila de subgrupo", [
    (["Solicitud hecha el 09/01/2025 por FREE", "", "", ""], True),
    (["25/02/2023", "AIR COMPUTER", "Sí", "1"], False),
    (["", "", "", ""], False),
], mi_es_subgrupo)

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_seccion_de(encabezados):
    claves = {clave_columna(e) for e in encabezados}
    if "informante" in claves:
        return "referencias"
    if "alertante" in claves:
        return "alertas"
    return None


def mi_es_subgrupo(celdas):
    con_texto = [c for c in celdas if c]
    # una sola celda con texto, y arranca con el prefijo conocido
    return len(con_texto) == 1 and celdas[0].startswith("Solicitud hecha el")
```


En el repo: [`pipeline/extractores/comun.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/pipeline/extractores/comun.py)

</details>

Con eso ya se puede armar el extractor completo. El del repo hace exactamente
esto más el streaming por páginas. Verificalo contra el ground truth:

In [ ]:
pdf.close()
correr(f"{PY} -m tests.verificar_extractor data/entrada/lote17.pdf | tail -3")

### La prueba de que el contrato servía

El repo trae un segundo extractor, de DOCX. Por dentro no se parecen en nada: uno
mide posiciones en la hoja, el otro recorre XML. Y esa misma celda combinada,
python-docx la devuelve **repetida en las 12 columnas** en vez de con `None`.

Si el contrato sirve, los dos tienen que producir lo mismo:

In [ ]:
correr(f"{PY} demo/generar_docx_fake.py | head -2")
cli("extraer data/entrada/lote17.pdf  -o /tmp/a.jsonl", mostrar=False)
cli("extraer data/entrada/lote17.docx -o /tmp/b.jsonl", mostrar=False)

import json
sin_origen = lambda p: [{k: v for k, v in json.loads(l).items() if k != "origen"}
                        for l in open(p, encoding="utf-8")]
print("\nJSONL desde PDF == JSONL desde DOCX:",
      sin_origen("/tmp/a.jsonl") == sin_origen("/tmp/b.jsonl"))

> ### 🤖 Pedíselo a Claude
>
> ```
> Escribí un extractor para este PDF que cumpla el contrato. Tres cosas
> a tener en cuenta: las fichas se derraman a varias páginas y las de continuación
> no repiten el encabezado, así que hay que streamear el documento entero y cortar
> por el encabezado; qué tabla es cuál hay que decidirlo por sus propios
> encabezados de columna y no por el título de sección; y las filas de celda
> combinada son títulos que hay que bajar a las filas siguientes.
> ```

---
# Paso 7 · Del texto crudo al dominio

El extractor devuelve las columnas **tal como se llaman en el documento**:
`"¿Es cliente?"`, `"Condición de venta"`, `"CT (U$D)"`. El dominio las traduce a
campos del modelo.

Ese mapeo es lo más frágil del pipeline: los encabezados cambian por detalles
cosméticos entre un documento y otro, o entre PDF y DOCX. Por eso se normalizan
antes de comparar.

## Tu turno

In [ ]:
COLUMNAS = {
    "fecha": "fecha",
    "informante": "informante",
    "es cliente": "es_cliente",
    "condicion de venta": "condicion_venta",
    "plazo": "plazo",
    "antiguedad": "antiguedad",
}

def mi_mapear(fila, columnas=COLUMNAS):
    """Traduce los encabezados del documento a campos del modelo.
    Las columnas que no estén en el mapeo se descartan."""
    # TODO: pista — clave_columna('¿Es cliente?') == 'es cliente'
    return {}

In [ ]:
checkpoint("Paso 7 · mapeo de columnas", [
    ({"Fecha": "12/5/2006", "¿Es cliente?": "Sí", "Condición de venta": "Contado"},
     {"fecha": "12/5/2006", "es_cliente": "Sí", "condicion_venta": "Contado"}),
    ({"Antigüedad": "2011", "Inactivo": "-"},
     {"antiguedad": "2011"}),
    ({"Columna Rara": "x"}, {}),
], mi_mapear)

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_mapear(fila, columnas=COLUMNAS):
    salida = {}
    for encabezado, valor in fila.items():
        campo = columnas.get(clave_columna(encabezado))
        if campo:
            salida[campo] = valor
    return salida
```


En el repo: [`pipeline/dominio/esquema.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/pipeline/dominio/esquema.py)

</details>

Fijate que `"Inactivo"` no está en el mapeo y desaparece. Eso es correcto: en
este dominio, `Inactivo = "Sí"` significa que **la fila entera se descarta**, así
que no hay ningún campo del modelo donde guardarlo.

Pero se cuenta aparte (`Cliente.descartadas`). Una exclusión legítima no es un
error, y aun así tenés que poder reconciliar: *"el documento tenía 76 filas, cargué
75, descarté 1"*.

---
# Paso 8 · La costura: JSONL y reconciliación

Etapa terminada, resultado a disco. Un cliente por línea.

No es capricho: es lo que te deja cortar un lote de 3000 y retomar, mirar el
resultado intermedio cuando algo no cierra, y reemplazar una etapa sin tocar las
demás.

In [ ]:
cli("extraer data/entrada/lote17.pdf")

**Lo primero que hay que leer es la reconciliación, no el "no hubo errores".**

`75 referencias + 1 descartada = 76 filas`. Ese número tiene que cerrar contra el
documento. Un parser puede terminar contento y haberse comido cuarenta filas.

In [ ]:
print(open("data/salida/lote17.jsonl", encoding="utf-8").readline()[:400], "...")

---
# Paso 9 · La revisión humana

Al humano no se lo saca del circuito: se lo **reubica**. En vez de copiar 300
fichas, mira una planilla y decide.

In [ ]:
cli("revisar data/salida/lote17.jsonl", mostrar=False)

from openpyxl import load_workbook
wb = load_workbook("data/salida/lote17.xlsx")
print("hojas:", wb.sheetnames, "\n")
print("PROBLEMAS (si está vacía, cargás tranquilo):")
for f in wb["Problemas"].iter_rows(values_only=True):
    print("  ", f)

Cuatro hojas, cuatro preguntas: **Resumen** (¿está todo el mundo?), **Detalle**
(¿este dato quedó bien?), **Alertas** (¿y los antecedentes?) y **Problemas**
(¿qué tengo que mirar?).

> ⚠️ **La flecha va en un solo sentido.** El JSONL es la fuente de verdad; el
> Excel es una foto. Las correcciones **no** vuelven editando el Excel: se corrige
> la regla en el código y se re-corre. Si el Excel fuera editable de vuelta, a la
> segunda corrida nadie sabría cuál de los dos tiene razón.

---
# Paso 10 · La clave, y cuándo NO cargar

Para cargar hay que poder decir **cuál fila de la web** le toca a cada fila del
documento. Eso es la clave.

Y tiene que salir del **documento**, no de la web: el id interno recién lo
conocés después de navegar hasta la fila, así que si la clave dependiera de él no
podrías saber qué saltear *antes* de ir a buscarlo.

Acá aparece la trampa más interesante del lote. El cliente STENFAR tiene dos
filas con **el mismo informante y la misma fecha** en la misma solicitud.

En la pantalla las dos matchean igual. ¿Qué hacés?

## Tu turno

In [ ]:
from collections import defaultdict

def mi_clave(cuit, ref):
    """Identifica una fila de referencia. ref tiene .solicitud, .fecha, .informante"""
    # TODO: fecha como dd/mm/aaaa; si no hay fecha, string vacío
    return ""


def mi_separar(items):
    """items: lista de (clave, dato).
    Devuelve (cargables, conflictos) — conflictos = claves repetidas."""
    # TODO
    return [], []

In [ ]:
checkpoint("Paso 10a · la clave", [
    (("30709204595", type("R", (), {"solicitud": "Sol X", "fecha": None,
                                    "informante": "ACME"})()),
     "30709204595|Sol X||ACME"),
], mi_clave)

print()
checkpoint("Paso 10b · separar conflictos", [
    ([("a", 1), ("b", 2), ("a", 3)], (["b"], ["a"])),
    ([("a", 1), ("b", 2)], (["a", "b"], [])),
], lambda items: tuple(sorted(x) if isinstance(x, list) else x
                       for x in mi_separar(items)))

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_clave(cuit, ref):
    fecha = ref.fecha.strftime("%d/%m/%Y") if ref.fecha else ""
    return f"{cuit}|{ref.solicitud}|{fecha}|{ref.informante}"


def mi_separar(items):
    por_clave = defaultdict(list)
    for clave, dato in items:
        por_clave[clave].append(dato)
    cargables = [c for c, g in por_clave.items() if len(g) == 1]
    conflictos = [c for c, g in por_clave.items() if len(g) > 1]
    return cargables, conflictos
```


En el repo: [`pipeline/carga/items.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/pipeline/carga/items.py)

</details>

In [ ]:
from pipeline import jsonl
from pipeline.carga import items as armador

clientes = list(jsonl.leer("data/salida/lote17.jsonl"))
cargables, conflictos = armador.armar(clientes)
print(f"cargables: {len(cargables)}   conflictos: {len(conflictos)}\n")
for c in conflictos:
    print("  ", c)

> **Automatizar bien no es cargar todo. Es saber qué no cargar.**

Elegir "la primera" escribiría un dato en la fila equivocada, y nadie se
enteraría nunca. Esas dos filas se reportan y quedan para un humano.

---
# Paso 11 · Las defensas

Cargar es la parte peligrosa: estás escribiendo en un sistema que no es tuyo.

El formulario de destino tiene campos que **el pipeline no debe tocar jamás**
(acá: Monto Asegurado, Seguro, Doc — y en alertas, el número de expediente).
Nada impide técnicamente escribirlos. Por eso hay una lista blanca.

## Tu turno

In [ ]:
LISTA_BLANCA = {"condicion", "credito_otorgado", "credito_tomado",
                "otorgado_usd", "tomado_usd", "antiguedad"}

def mi_validar(campos, lista_blanca=LISTA_BLANCA):
    """Devuelve [] si todo OK, o la lista ordenada de campos prohibidos."""
    # TODO
    return []

In [ ]:
checkpoint("Paso 11 · lista blanca", [
    ({"condicion": "Contado", "antiguedad": "12/05/2006"}, []),
    ({"condicion": "Contado", "monto_asegurado": "999"}, ["monto_asegurado"]),
    ({"seguro": "SI", "doc": "A-1"}, ["doc", "seguro"]),
], mi_validar)

<details>
<summary><b>🔑 Si te trabaste — abrir la solución</b></summary>

```python
def mi_validar(campos, lista_blanca=LISTA_BLANCA):
    return sorted(set(campos) - set(lista_blanca))
```


En el repo: [`pipeline/carga/navegador.py`](https://github.com/GEJ1/data_entry_automatizado/blob/main/pipeline/carga/navegador.py)

</details>

Y un campo fuera de la lista **aborta la fila entera**, sin abrir el navegador.
No se escribe "lo que se pueda":

In [ ]:
from pipeline.carga.mapeo import Mapeo
from pipeline.carga.navegador import CargadorNulo
from pipeline.contratos import ItemDeCarga

mapeo = Mapeo.cargar("config/mapeo_web.yaml")
malo = ItemDeCarga(formulario="referencias", clave="demo", busqueda={"cuit": "x"},
                   campos={"condicion": "Contado", "monto_asegurado": "999"})
with CargadorNulo(mapeo) as c:
    print(c.cargar(malo).detalle)

### Las otras dos defensas

La lista blanca sola es una promesa. Estas dos la hacen **verificable**:

- **Nada más se movió** — se fotografían *todos* los inputs del formulario antes
  y después de guardar. Si cambió algo que no estaba en la lista, la fila se
  reporta como error aunque el guardado haya funcionado. Como se fotografía todo
  y no una lista de nombres, un campo nuevo que agreguen mañana también queda
  protegido.
- **Read-back** — después de guardar se reabre el formulario y se compara contra
  lo que se quiso escribir. Que el submit no explote no significa que el dato entró.

### Y la idempotencia

El estado de cada fila va a un SQLite. Un lote de 3000 tarda y **se va a cortar**:
se cae la red, vence la sesión, cierran la notebook. Sin esto, retomar significa
empezar de cero y arriesgarse a cargar dos veces.

> Una sutileza que fue un bug real en este proyecto: **`--dry-run` no anota
> estado**. Si anotara, la corrida real saltearía filas que nunca se escribieron
> y quedarían vacías para siempre.

> ### 🤖 Pedíselo a Claude
>
> ```
> Escribí el cargador con Playwright. Los selectores y las URLs tienen
> que venir de un archivo de configuración, no estar en el código. Agregale tres
> defensas: una lista blanca de campos escribibles (un campo fuera de la lista
> aborta la fila sin tocar la web), una comparación de todos los inputs del
> formulario antes y después de guardar para detectar si se modificó algo fuera de
> la lista, y un read-back que reabra el formulario y verifique que el dato entró.
> El estado de cada fila en SQLite para poder cortar y retomar.
> ```

---
# Paso 12 · La carga · ⚠️ esto va en tu terminal

**Este paso no se corre en el cuaderno**, y no es un capricho técnico: es el
único momento del curso donde **ves la automatización trabajando**. Un navegador
buscando el CUIT, entrando a la solicitud, ubicando la fila, llenando los campos.

Metido dentro de una celda lo verías como una captura estática, que no es lo
mismo.

> Si estás en Colab, este paso no se puede hacer. Cloná el repo en tu máquina:
> ```
> git clone https://github.com/GEJ1/data_entry_automatizado.git
> cd data_entry_automatizado
> python3.10 -m venv .venv && source .venv/bin/activate
> pip install -r requirements.txt && playwright install chromium
> python -m pipeline.cli extraer data/entrada/lote17.pdf
> ```

### Terminal 1 — la web de destino

```bash
python web_demo/sembrar.py data/salida/lote17.jsonl
python web_demo/app.py
```

Abrila en http://127.0.0.1:5000 y **hacé una fila a mano primero**: buscá el CUIT
`33645428418`, entrá a la solicitud, tocá el lápiz, completá y guardá. Eso es lo
que hace la persona todos los días. Ahora lo vas a automatizar.

Fijate que la URL es `/admin/solicitudes/1/financieras/1/edit` y **ninguno de esos
dos números está en el documento**. No se pueden calcular: hay que buscar,
navegar y cosechar el href del lápiz.

### Terminal 2 — la carga, mirándola

```bash
# dry-run: llena el formulario y NO guarda
python -m pipeline.cli cargar data/salida/lote17.jsonl --dry-run --limite 3 --ver --lento 400
```

- `--ver` abre Chromium con ventana en vez de headless
- `--lento 400` mete 400 ms entre acción y acción, si no va demasiado rápido para
  seguirlo con la vista
- `--dry-run` llena todo y **no guarda**: siempre esta primero

Cuando te convenza, la de verdad:

```bash
python -m pipeline.cli cargar data/salida/lote17.jsonl
```

Y para ver la idempotencia: cortala con **Ctrl-C** a la mitad y volvé a correrla.

## Verificar qué pasó

Corré esta celda **después** de haber hecho la carga en la terminal.

In [ ]:
import sqlite3
from pathlib import Path

VACIO = "''"   # comparar contra string vacío en SQL

if not Path("web_demo/datos.db").exists():
    print("Todavía no sembraste la web. En tu máquina:")
    print("   python web_demo/sembrar.py data/salida/lote17.jsonl")
else:
    con = sqlite3.connect("web_demo/datos.db")
    q = lambda s: con.execute(s).fetchone()[0]

    refs_ok  = q(f"SELECT COUNT(*) FROM financieras WHERE condicion != {VACIO}")
    refs_tot = q("SELECT COUNT(*) FROM financieras")
    ale_ok   = q(f"SELECT COUNT(*) FROM alertas WHERE tipo != {VACIO}")
    ale_tot  = q("SELECT COUNT(*) FROM alertas")

    print(f"referencias cargadas : {refs_ok} de {refs_tot}")
    print(f"alertas cargadas     : {ale_ok} de {ale_tot}")

    if refs_ok == 0:
        print("\n→ Todavía no cargaste nada. Corré la carga en la terminal.")
    else:
        pisadas = q(f"SELECT COUNT(*) FROM financieras WHERE monto_asegurado = {VACIO}")
        print(f"pólizas pisadas      : {pisadas}   (tiene que ser 0)")
        print("\nlas filas en conflicto, que NO se cargaron:")
        for f in con.execute(
                "SELECT fecha, informante, condicion FROM financieras "
                "WHERE informante = 'MICROGLOBAL' AND fecha = '02/06/2026'"):
            print("   ", f, "← vacías: el pipeline se negó a adivinar")
    con.close()

---
# Lo que construiste

```
archivo ──[Extractor]──→ FichaCruda ──[Dominio]──→ Cliente ──→ .jsonl ──→ ItemDeCarga ──[Cargador]──→ web
```

Con tres defensas verificadas, idempotencia, y una verificación automática contra
ground truth.

**Y no es un cargador de solicitudes financieras: es un esqueleto con dos
enchufes.** El caso que usamos fue una excusa.

## Ahora, lo tuyo

En **`03_tu_caso.ipynb`** lo apuntamos a tu archivo y tu web.

| Qué cambia | Qué tocás |
|---|---|
| Otro **formato** de entrada | una clase en `pipeline/extractores/` + una línea en el registro |
| Otro **rubro** | `pipeline/dominio/esquema.py` (los normalizadores sobreviven) |
| Otra **web** | `config/mapeo_web.yaml` |
| Otro **formulario** en la misma web | un bloque en el YAML + un `elif` en `_ir_a_la_tabla` |